# AI-Based Hospital Readmission Prediction System

# 2. Data Preprocessing

# Introduction

 This notebook prepares the hospital readmission dataset for machine learning.

 The main preprocessing steps include:

 - Creating a binary target variable
 - Removing identifier columns
 - Handling missing values
 - Splitting data into training and testing sets
 - Identifying numerical and categorical features
 - Imputing missing values
 - Encoding categorical variables
 - Scaling numerical variables
 - Creating a reproducible preprocessing pipeline
 - Saving preprocessing artifacts

  The positive class is defined as:

 - 1 = Patient readmitted within 30 days
 - 0 = Patient not readmitted within 30 days

In [1]:

# It is used to create folders and manage file paths.
import os

# Import joblib
# It is used to save and load machine learning objects.
import joblib

# Import NumPy
# It is used for numerical operations and handling missing values.
import numpy as np

# Import Pandas
# It is used for loading, cleaning, and manipulating tabular data.
import pandas as pd

# Import train_test_split
# It is used to divide the dataset into training and testing sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer
# It allows different preprocessing techniques for different column types.
from sklearn.compose import ColumnTransformer

# Import Pipeline
# It combines multiple preprocessing steps into one workflow.
from sklearn.pipeline import Pipeline

# Import SimpleImputer
# It is used to fill missing values.
from sklearn.impute import SimpleImputer

# Import OneHotEncoder
# It converts categorical variables into numerical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler
# It standardizes numerical variables to a common scale.
from sklearn.preprocessing import StandardScaler

# Display all columns when viewing DataFrames
pd.set_option("display.max_columns", None)

# Display numerical values with two decimal places
pd.set_option(
    "display.float_format",
    lambda x: f"{x:.2f}"
)

# Confirm successful library import
print("All required libraries imported successfully.")

All required libraries imported successfully.


# 1. Load the Dataset

 The original hospital readmission dataset is loaded in this step.

 The raw CSV file is not modified. Any preprocessing operation is performed
 only on the DataFrame used in this notebook.

In [2]:
# Load the original hospital readmission dataset
df = pd.read_csv("diabetic_data.csv")

# Replace '?' with NaN
# In this dataset, '?' represents an unknown or missing value.
df = df.replace("?", np.nan)

# Display the dataset dimensions
print("Dataset loaded successfully.")

# Print the number of rows and columns
print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

Dataset loaded successfully.
Number of rows    : 101766
Number of columns : 50


# 2. Create the Binary Target Variable

 The original `readmitted` column contains three classes:

 - `<30` = Patient was readmitted within 30 days
 - `>30` = Patient was readmitted after 30 days
 - `NO` = Patient was not readmitted

 For this binary classification problem:

 - 1 = Readmitted within 30 days
 - 0 = Not readmitted within 30 days

In [3]:
# Create a binary target variable named readmitted_binary
# The value becomes 1 when readmission occurred within 30 days.
# All other readmission categories become 0.
df["readmitted_binary"] = (
    df["readmitted"] == "<30"
).astype(int)

# Count the number of records in each binary target class
target_counts = df["readmitted_binary"].value_counts()

# Display the target distribution
print("Binary target distribution:")
print(target_counts)

# Calculate the percentage of each target class
target_percentage = (
    df["readmitted_binary"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Display the target percentages
print("\nBinary target percentage:")
print(target_percentage)

Binary target distribution:
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

Binary target percentage:
readmitted_binary
0   88.84
1   11.16
Name: proportion, dtype: float64


# 3. Remove Identifier and Original Target Columns

 Patient identifiers are not useful predictive features and should not be used
 for model training.

 The original `readmitted` column is also removed because the binary target
 `readmitted_binary` has already been created.

 This helps prevent target leakage and unnecessary information from entering
 the machine learning model.

In [4]:
# Define the identifier columns that should not be used as model features
id_columns = [
    "encounter_id",
    "patient_nbr"
]

# Define the columns that need to be removed
# The original target and patient identifiers are excluded.
columns_to_remove = id_columns + ["readmitted"]

# Create a model-ready DataFrame without the unwanted columns
df_model = df.drop(
    columns=columns_to_remove,
    errors="ignore"
)

# Display the columns that were removed
print("Removed columns:")
for column in columns_to_remove:
    print(f"- {column}")

# Display the new dataset shape
print("\nDataset shape after removing unnecessary columns:")
print(df_model.shape)

Removed columns:
- encounter_id
- patient_nbr
- readmitted

Dataset shape after removing unnecessary columns:
(101766, 48)


# 4. Missing-Value Analysis

 We calculate the number and percentage of missing values in each feature.

 This helps us understand the quality of the available data and identify
 columns that contain a very large amount of missing information.

In [5]:
# Count missing values in every column
missing_count = df_model.isnull().sum()

# Calculate the percentage of missing values in every column
missing_percentage = (
    df_model.isnull().mean() * 100
).round(2)

# Combine missing count and percentage into one DataFrame
missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

# Keep only columns that contain at least one missing value
missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
]

# Sort the result from highest to lowest missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

# Display the missing-value summary
missing_summary

,Missing_Count,Missing_Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


# 5. Remove Extremely Sparse Columns

 Columns with more than 90% missing values contain very limited information.

 Such columns are removed because they provide insufficient information for
 reliable machine learning analysis.

 The threshold used here is 90%.

In [7]:
# Define the maximum allowed missing-value percentage
missing_threshold = 90

# Calculate the missing percentage for every column
column_missing_percentage = (
    df_model.isnull().mean() * 100
)

# Identify columns where more than 90% of values are missing
high_missing_columns = (
    column_missing_percentage[
        column_missing_percentage > missing_threshold
    ]
    .index
    .tolist()
)

# Display the identified columns
print("Columns with more than 90% missing values:")

if len(high_missing_columns) == 0:
    # Display this message when no column crosses the threshold
    print("No columns found.")
else:
    # Display each highly missing column
    for column in high_missing_columns:
        print(f"- {column}")

# Display the total number of columns identified
print(
    f"\nNumber of highly missing columns: "
    f"{len(high_missing_columns)}"
)

Columns with more than 90% missing values:
- weight
- max_glu_serum

Number of highly missing columns: 2


In [8]:
# Remove the columns containing more than 90% missing values
df_model = df_model.drop(
    columns=high_missing_columns,
    errors="ignore"
)

# Display the dataset shape after removing sparse columns
print("Dataset shape after removing highly missing columns:")
print(df_model.shape)

Dataset shape after removing highly missing columns:
(101766, 46)


# 6. Separate Features and Target

 The dataset is divided into two parts:

# X — Features
 Contains the patient and healthcare-related variables used for prediction.

# y — Target
 Contains the binary readmission outcome.

In [9]:
# Create the feature dataset by removing the target column
X = df_model.drop(
    columns=["readmitted_binary"]
)

# Store the binary target separately
y = df_model["readmitted_binary"]

# Display the dimensions of the feature dataset
print(f"Feature dataset shape: {X.shape}")

# Display the dimensions of the target dataset
print(f"Target dataset shape : {y.shape}")

# Confirm the target column is not present in X
print(
    "\nIs target present in feature dataset?",
    "readmitted_binary" in X.columns
)

Feature dataset shape: (101766, 45)
Target dataset shape : (101766,)

Is target present in feature dataset? False


# 7. Train-Test Split

 The dataset is divided into:

 - 80% training data
 - 20% testing data

 The training data is used to train the machine learning models.

 The testing data is kept separate and is used later for unbiased model
 evaluation.

 Stratification is used so that both training and testing datasets maintain
 approximately the same proportion of positive and negative target classes.

In [10]:
# Split the feature and target data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the training dataset dimensions
print("Training feature shape:")
print(X_train.shape)

# Display the testing dataset dimensions
print("\nTesting feature shape:")
print(X_test.shape)

# Display the training target distribution
print("\nTraining target distribution:")
print(
    y_train.value_counts(
        normalize=True
    ).round(4)
)

# Display the testing target distribution
print("\nTesting target distribution:")
print(
    y_test.value_counts(
        normalize=True
    ).round(4)
)

Training feature shape:
(81412, 45)

Testing feature shape:
(20354, 45)

Training target distribution:
readmitted_binary
0   0.89
1   0.11
Name: proportion, dtype: float64

Testing target distribution:
readmitted_binary
0   0.89
1   0.11
Name: proportion, dtype: float64


# 8. Identify Numerical and Categorical Features

 Machine learning models require numerical input.

 Therefore, we separate the features into two groups:

# Numerical Features
 These contain numeric values such as hospital stay, medications,
 procedures and previous visits.

# Categorical Features
 These contain categories such as gender, race, age group and medication
 status.

 Different preprocessing techniques will be applied to these two groups.

In [11]:
# Identify numerical columns from the training dataset
# These columns contain integer or floating-point values.
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical columns from the training dataset
# These columns contain text/category values.
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Display the number of features in each group
print(
    f"Number of numerical features   : "
    f"{len(numerical_columns)}"
)

print(
    f"Number of categorical features : "
    f"{len(categorical_columns)}"
)

Number of numerical features   : 11
Number of categorical features : 34


In [12]:
# Display the names of all numerical features
print("Numerical features:")

# Loop through every numerical feature
for i, column in enumerate(
    numerical_columns,
    start=1
):
    # Print the feature number and name
    print(f"{i}. {column}")

Numerical features:
1. admission_type_id
2. discharge_disposition_id
3. admission_source_id
4. time_in_hospital
5. num_lab_procedures
6. num_procedures
7. num_medications
8. number_outpatient
9. number_emergency
10. number_inpatient
11. number_diagnoses


In [13]:
# Display the names of all categorical features
print("Categorical features:")

# Loop through every categorical feature
for i, column in enumerate(
    categorical_columns,
    start=1
):
    # Print the feature number and name
    print(f"{i}. {column}")

Categorical features:
1. race
2. gender
3. age
4. payer_code
5. medical_specialty
6. diag_1
7. diag_2
8. diag_3
9. A1Cresult
10. metformin
11. repaglinide
12. nateglinide
13. chlorpropamide
14. glimepiride
15. acetohexamide
16. glipizide
17. glyburide
18. tolbutamide
19. pioglitazone
20. rosiglitazone
21. acarbose
22. miglitol
23. troglitazone
24. tolazamide
25. examide
26. citoglipton
27. insulin
28. glyburide-metformin
29. glipizide-metformin
30. glimepiride-pioglitazone
31. metformin-rosiglitazone
32. metformin-pioglitazone
33. change
34. diabetesMed


# 9. Numerical Preprocessing Pipeline

 Numerical features may contain missing values and may have different scales.

 We use two preprocessing steps:

 1. Median Imputation
    - Missing numerical values are replaced with the median.

 2. Standard Scaling
    - Numerical features are transformed to a comparable scale.

 The pipeline ensures that these operations are applied consistently.

In [14]:
# Create the numerical preprocessing pipeline
numerical_pipeline = Pipeline(
    steps=[
        # Replace missing numerical values with the median
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        # Standardize numerical features
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Confirm successful pipeline creation
print(
    "Numerical preprocessing pipeline "
    "created successfully."
)

Numerical preprocessing pipeline created successfully.


# 10. Categorical Preprocessing Pipeline

 Categorical variables also require preprocessing.

 We use two steps:

 1. Most-Frequent Imputation
    - Missing categorical values are replaced with the most common category.

 2. One-Hot Encoding
    - Categories are converted into numerical columns.

 `handle_unknown="ignore"` prevents errors when the test dataset contains a
 category that was not present in the training dataset.

In [15]:
# Create the categorical preprocessing pipeline
categorical_pipeline = Pipeline(
    steps=[
        # Replace missing categorical values
        # with the most frequently occurring category
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        # Convert categorical values into numerical columns
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Confirm successful pipeline creation
print(
    "Categorical preprocessing pipeline "
    "created successfully."
)

Categorical preprocessing pipeline created successfully.


# 11. Combine the Preprocessing Pipelines

 The numerical and categorical preprocessing pipelines are combined using
 `ColumnTransformer`.

 This creates one complete preprocessing system where:

 - Numerical columns receive numerical preprocessing.
 - Categorical columns receive categorical preprocessing.

In [16]:
# Create a ColumnTransformer to combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        # Apply numerical preprocessing
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        ),

        # Apply categorical preprocessing
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

# Confirm successful creation
print(
    "Combined preprocessing pipeline "
    "created successfully."
)

Combined preprocessing pipeline created successfully.


# 12. Fit the Preprocessing Pipeline

 The preprocessing pipeline is fitted only on the training data.

 This is important because information from the testing dataset should not be
 used while learning preprocessing parameters.

 After fitting on training data, the same transformation is applied to the
 testing data.

In [17]:
# Fit the preprocessing pipeline using only training data
X_train_processed = preprocessor.fit_transform(
    X_train
)

# Transform the testing data using the already-fitted pipeline
X_test_processed = preprocessor.transform(
    X_test
)

# Confirm that preprocessing was completed
print(
    "Preprocessing completed successfully."
)

# Display the processed training data shape
print(
    f"\nProcessed training data shape: "
    f"{X_train_processed.shape}"
)

# Display the processed testing data shape
print(
    f"Processed testing data shape : "
    f"{X_test_processed.shape}"
)

Preprocessing completed successfully.

Processed training data shape: (81412, 2358)
Processed testing data shape : (20354, 2358)


# 13. Verify the Preprocessed Data

 The processed dataset should contain numerical values that can be provided
 to machine learning algorithms.

 We verify the data type and dimensions of the transformed datasets.

In [18]:
# Display the Python data type of processed training data
print("Processed training data type:")
print(type(X_train_processed))

# Display the shape of processed training data
print("\nProcessed training data shape:")
print(X_train_processed.shape)

# Display the shape of processed testing data
print("\nProcessed testing data shape:")
print(X_test_processed.shape)

# Check whether the training dataset contains any NaN values
print(
    "\nTraining data contains NaN:",
    np.isnan(
        X_train_processed.toarray()
        if hasattr(X_train_processed, "toarray")
        else X_train_processed
    ).any()
)

Processed training data type:
<class 'scipy.sparse._csr.csr_matrix'>

Processed training data shape:
(81412, 2358)

Processed testing data shape:
(20354, 2358)

Training data contains NaN: False


# 14. Get Processed Feature Names

 One-Hot Encoding creates additional columns from categorical variables.

 We retrieve the names of all features generated by the preprocessing pipeline.

 These feature names will also be useful during model interpretation and
 feature-importance analysis.

In [19]:
# Retrieve the names of all transformed features
feature_names = preprocessor.get_feature_names_out()

# Display the total number of transformed features
print(
    f"Total transformed features: "
    f"{len(feature_names)}"
)

# Display the first 20 transformed feature names
print("\nFirst 20 transformed features:")

for i, feature in enumerate(
    feature_names[:20],
    start=1
):
    # Print feature number and name
    print(f"{i}. {feature}")

Total transformed features: 2358

First 20 transformed features:
1. numerical__admission_type_id
2. numerical__discharge_disposition_id
3. numerical__admission_source_id
4. numerical__time_in_hospital
5. numerical__num_lab_procedures
6. numerical__num_procedures
7. numerical__num_medications
8. numerical__number_outpatient
9. numerical__number_emergency
10. numerical__number_inpatient
11. numerical__number_diagnoses
12. categorical__race_AfricanAmerican
13. categorical__race_Asian
14. categorical__race_Caucasian
15. categorical__race_Hispanic
16. categorical__race_Other
17. categorical__gender_Female
18. categorical__gender_Male
19. categorical__gender_Unknown/Invalid
20. categorical__age_[0-10)


# 15. Create Artifacts Folder

 Important preprocessing objects will be saved inside an `artifacts` folder.

 These saved objects will later be reused during:

 - Model training
 - Model evaluation
 - Deployment

 This ensures that the same preprocessing logic is used throughout the
 complete machine learning lifecycle.

In [20]:
# Create the artifacts directory
# exist_ok=True prevents an error if the folder already exists.
os.makedirs(
    "artifacts",
    exist_ok=True
)

# Save the fitted preprocessing pipeline
joblib.dump(
    preprocessor,
    "artifacts/preprocessor.pkl"
)

# Save the names of transformed features
joblib.dump(
    feature_names,
    "artifacts/feature_names.pkl"
)

# Save the original model input column names
joblib.dump(
    X.columns.tolist(),
    "artifacts/model_input_columns.pkl"
)

# Confirm that the preprocessing objects were saved
print(
    "Preprocessing artifacts saved successfully."
)

Preprocessing artifacts saved successfully.


# 16. Save Train-Test Data

 The training and testing datasets are saved so that the same data split can
 be reused during model training and evaluation.

 The target variables are saved separately from the feature datasets.

In [21]:
# Save training features
joblib.dump(
    X_train,
    "artifacts/X_train.pkl"
)

# Save testing features
joblib.dump(
    X_test,
    "artifacts/X_test.pkl"
)

# Save training target
joblib.dump(
    y_train,
    "artifacts/y_train.pkl"
)

# Save testing target
joblib.dump(
    y_test,
    "artifacts/y_test.pkl"
)

# Confirm successful saving
print(
    "Train-test datasets saved successfully."
)

Train-test datasets saved successfully.


# 17. Final Preprocessing Summary

 The data preprocessing stage has been completed.

# Completed Steps

 - Loaded the original dataset
 - Converted `?` values into `NaN`
 - Created a binary readmission targe
- Removed patient identifier columns
 - Removed the original target column
 - Removed extremely sparse columns
 - Split the data into training and testing sets
 - Used stratified sampling
 - Identified numerical and categorical features
 - Applied median imputation to numerical features
 - Applied most-frequent imputation to categorical features
 - Applied StandardScaler to numerical features
 - Applied One-Hot Encoding to categorical features
 - Created a reproducible preprocessing pipeline
 - Saved preprocessing artifacts

 The prepared data is now ready for exploratory analysis and feature
 engineering.

In [23]:
# Print a final summary of the preprocessing stage

print("=" * 60)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 60)

# Display the original dataset size
print(
    f"Original dataset shape      : "
    f"{df.shape}"
)

# Display the model dataset size
print(
    f"Model dataset shape         : "
    f"{df_model.shape}"
)

# Display the number of training samples
print(
    f"Training samples            : "
    f"{X_train.shape[0]}"
)

# Display the number of testing samples
print(
    f"Testing samples             : "
    f"{X_test.shape[0]}"
)

# Display the number of original input features
print(
    f"Original input features     : "
    f"{X.shape[1]}"
)

# Display the number of transformed features
print(
    f"Transformed features        : "
    f"{X_train_processed.shape[1]}"
)

# Display the number of numerical features
print(
    f"Numerical features          : "
    f"{len(numerical_columns)}"
)

# Display the number of categorical features
print(
    f"Categorical features        : "
    f"{len(categorical_columns)}"
)

# Display the location of saved artifacts
print("\nSaved artifacts location:")
print("artifacts/")

# Final completion message
print("\nData Preprocessing completed successfully.")

DATA PREPROCESSING COMPLETED SUCCESSFULLY
Original dataset shape      : (101766, 51)
Model dataset shape         : (101766, 46)
Training samples            : 81412
Testing samples             : 20354
Original input features     : 45
Transformed features        : 2358
Numerical features          : 11
Categorical features        : 34

Saved artifacts location:
artifacts/

Data Preprocessing completed successfully.
